# Silver: PDFs to Markdown

One table, `silver_pdfs`, filled in separate cells:

```
bronze_pdfs --ai_parse_document--> silver_pdfs.parsed --ai_query--> silver_pdfs.markdown
```

Incremental: step 2 streams from bronze with a checkpoint, and step 3 only converts rows with no markdown yet.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("endpoint", "databricks-claude-sonnet-4")
dbutils.widgets.text("max_files_per_batch", "10")  # step 2: bronze files per micro-batch
dbutils.widgets.text("convert_per_run", "20")      # step 3: documents converted per run

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
endpoint = dbutils.widgets.get("endpoint")
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))
convert_per_run = int(dbutils.widgets.get("convert_per_run"))

bronze = f"{catalog}.{schema}.bronze_pdfs"
silver = f"{catalog}.{schema}.silver_pdfs"
checkpoint = f"/Volumes/{catalog}/{schema}/docs/_checkpoints/silver_pdfs"

## 1. Create the table

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver} (
  path STRING,
  parsed VARIANT,
  markdown STRING
)
""")

## 2. Parse with `ai_parse_document` (streaming)

Reads `bronze_pdfs` as a stream and appends the parsed output to `silver_pdfs`. `maxFilesPerTrigger` keeps each micro-batch small, so progress is committed often and a failure only loses the current batch. The checkpoint remembers which bronze rows were already processed, so each run parses only new PDFs. `trigger(availableNow=True)` processes everything new and stops. `markdown` stays empty until step 3.

In [ ]:
(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).table(bronze)
    .select(
        "path",
        F.expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed"),
        F.lit(None).cast("string").alias("markdown"),
    )
    .writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(silver)
    .awaitTermination()
)
print(spark.table(silver).count(), "parsed")

## 3. Convert to Markdown with `ai_query`

Fills `markdown` for up to `convert_per_run` rows where it is still empty. A failure then loses at most one run's worth of calls. Rerun until the count of empty rows is 0.

In [ ]:
PROMPT = """Convert this parsed PDF (JSON from ai_parse_document) into Markdown.

Rules:
- Keep ALL content exactly as written. Do not summarize, reword or invent anything.
- Titles become "# Title"; section headers become "##" / "###".
- Keep plain text as paragraphs and lists as Markdown lists.
- Convert tables to Markdown tables (header row, | --- | separator row), keeping every row and column.
- Keep metadata such as page headers, footers and page numbers as italic lines.
- Put a page separator between pages, exactly: a blank line, "---", then "<!-- page N -->" (N = page number), then a blank line. Start the document with "<!-- page 1 -->".
- Output ONLY the Markdown, with no code fences or commentary.

Parsed document:
"""

pending = (
    spark.table(silver).filter("markdown IS NULL").limit(convert_per_run)
    .withColumn("prompt", F.concat(F.lit(PROMPT), F.col("parsed").cast("string")))
    .select(
        "path",
        F.expr(
            f"ai_query('{endpoint}', prompt, "
            "modelParameters => named_struct('temperature', 0.0, 'max_tokens', 32000))"
        ).alias("markdown"),
    )
)

(
    DeltaTable.forName(spark, silver).alias("t")
    .merge(pending.alias("s"), "t.path = s.path")
    .whenMatchedUpdate(set={"markdown": "s.markdown"})
    .execute()
)
print(spark.table(silver).filter("markdown IS NOT NULL").count(), "converted,",
      spark.table(silver).filter("markdown IS NULL").count(), "left")

## 4. Check

In [ ]:
r = spark.table(silver).filter("markdown IS NOT NULL").limit(1).collect()
if r:
    print(r[0]["path"]); print(r[0]["markdown"][:2000])

## Notes

- Full rebuild: drop `silver_pdfs`, delete the checkpoint folder (`/Volumes/<catalog>/<schema>/docs/_checkpoints/silver_pdfs`) and rerun. Dropping only the table leaves the checkpoint, so nothing would be reparsed. To re-run only the prompt: `UPDATE silver_pdfs SET markdown = NULL`, then rerun step 3.
- Set the `endpoint` parameter to a Foundation Model endpoint in your workspace, with a large context window.
- `ai_parse_document` and `ai_query` are billed per use. Test on a few PDFs first.